# Movie Recommendation System
### Data Science Lab — Mini Project (Experiment 10)

**Aim:** To develop a movie recommendation system using Python and machine learning techniques to recommend movies based on content similarity.

**Approach:** This project implements a **content-based recommendation system**. It combines movie genres, overview, keywords, and cast/director information into one text description, converts that text into numerical vectors using **TF-IDF**, and ranks movies using **cosine similarity**.

**Learning outcomes**
- Prepare and inspect a movie dataset.
- Convert text features into TF-IDF vectors.
- Calculate cosine similarity between movies.
- Recommend movies similar to a user's selected title.

> The notebook includes a small sample dataset so it can run immediately without downloading a dataset. For more meaningful recommendations, replace it with a larger movie CSV using the optional section near the end.


## 1. Import required libraries

We use:
- `pandas` to store and process movie data.
- `scikit-learn` to create TF-IDF vectors and calculate cosine similarity.
- `re` to clean text and `difflib` to suggest close title matches.


In [9]:
import re
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the movie dataset

This sample dataset is embedded in the notebook, so no separate file is required. Each movie has a title, genres, overview, keywords, cast, and director. These fields are used to identify movies with similar content.


In [33]:
df = pd.read_csv("./movie_recommendation_dataset.csv")

print(f"Dataset contains {len(df)} movies.")
# display(df[["Title", "Genre", "Director"]].head(10))
display(df.head(10))

Dataset contains 4800 movies.


,title,genres,overview,keywords,cast,director,release_date,vote_average,vote_count
0,Avatar,Action Adventure Fantasy Science Fiction,"In the 22nd century, a paraplegic Marine is di...",culture clash future space war space colony so...,Sam Worthington Zoe Saldana Sigourney Weaver S...,James Cameron,2009-12-10,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,"Captain Barbossa, long believed to be dead, ha...",ocean drug abuse exotic island east india trad...,Johnny Depp Orlando Bloom Keira Knightley Stel...,Gore Verbinski,2007-05-19,6.9,4500
2,Spectre,Action Adventure Crime,A cryptic message from Bond’s past sends him o...,spy based on novel secret agent sequel mi6 bri...,Daniel Craig Christoph Waltz Léa Seydoux Ralph...,Sam Mendes,2015-10-26,6.3,4466
3,The Dark Knight Rises,Action Crime Drama Thriller,Following the death of District Attorney Harve...,dc comics crime fighter terrorist secret ident...,Christian Bale Michael Caine Gary Oldman Anne ...,Christopher Nolan,2012-07-16,7.6,9106
4,John Carter,Action Adventure Science Fiction,"John Carter is a war-weary, former military ca...",based on novel mars medallion space travel pri...,Taylor Kitsch Lynn Collins Samantha Morton Wil...,Andrew Stanton,2012-03-07,6.1,2124
5,Spider-Man 3,Fantasy Action Adventure,The seemingly invincible Spider-Man goes up ag...,dual identity amnesia sandstorm love of one's ...,Tobey Maguire Kirsten Dunst James Franco Thoma...,Sam Raimi,2007-05-01,5.9,3576
6,Tangled,Animation Family,When the kingdom's most wanted-and most charmi...,hostage magic horse fairy tale musical princes...,Zachary Levi Mandy Moore Donna Murphy Ron Perl...,Byron Howard,2010-11-24,7.4,3330
7,Avengers: Age of Ultron,Action Adventure Science Fiction,When Tony Stark tries to jumpstart a dormant p...,marvel comic sequel superhero based on comic b...,Robert Downey Jr. Chris Hemsworth Mark Ruffalo...,Joss Whedon,2015-04-22,7.3,6767
8,Harry Potter and the Half-Blood Prince,Adventure Fantasy Family,"As Harry begins his sixth year at Hogwarts, he...",witch magic broom school of witchcraft wizardr...,Daniel Radcliffe Rupert Grint Emma Watson Tom ...,David Yates,2009-07-07,7.4,5293
9,Batman v Superman: Dawn of Justice,Action Adventure Fantasy,Fearing the actions of a god-like Super Hero l...,dc comics vigilante superhero based on comic b...,Ben Affleck Henry Cavill Gal Gadot Amy Adams J...,Zack Snyder,2016-03-23,5.7,7004


## 3. Inspect and prepare the data

Missing values in the text fields are replaced with empty strings. We then combine each movie's genres, overview, keywords, cast, and director into a single text field called `combined_features`.


In [35]:
text_columns = ["genres", "overview", "keywords", "cast", "director"]

for column in ["title"] + text_columns:
    df[column] = df[column].fillna("").astype(str)

def clean_text(text):
    # Lowercase and keep letters/numbers separated by spaces
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["combined_features"] = df[text_columns].agg(" ".join, axis=1).apply(clean_text)

display(df[["title", "combined_features"]].head(3))

,title,combined_features
0,Avatar,action adventure fantasy science fiction in th...
1,Pirates of the Caribbean: At World's End,adventure fantasy action captain barbossa long...
2,Spectre,action adventure crime a cryptic message from ...


## 4. Convert text into TF-IDF vectors

TF-IDF gives greater importance to words that help distinguish one movie from others. `stop_words="english"` removes common English words. The resulting matrix represents each movie numerically.


In [36]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Rows = movies; columns = terms in the vocabulary.")

TF-IDF matrix shape: (4800, 30737)
Rows = movies; columns = terms in the vocabulary.


## 5. Calculate cosine similarity

Cosine similarity measures the angle between two vectors. Scores closer to 1 indicate more similar text features, while scores closer to 0 indicate less similarity. We calculate the similarity of every movie with every other movie.


In [37]:
similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df["title"],
    columns=df["title"]
)

display(similarity_df.round(2).iloc[:5, :5])

title,Avatar,Pirates of the Caribbean: At World's End,Spectre,The Dark Knight Rises,John Carter
title,,,,,
Avatar,1.00,0.02,0.02,0.01,0.16
Pirates of the Caribbean: At World's End,0.02,1.00,0.01,0.01,0.03
Spectre,0.02,0.01,1.00,0.02,0.02
The Dark Knight Rises,0.01,0.01,0.02,1.00,0.01
John Carter,0.16,0.03,0.02,0.01,1.00


## 6. Build the recommendation function

The function:
1. Finds the selected movie.
2. Gets its similarity scores with all other movies.
3. Sorts movies from most to least similar.
4. Excludes the selected movie itself.
5. Returns the top recommendations.

If the title is not an exact match, the function suggests a similar title.


In [39]:
from difflib import get_close_matches

def recommend_movies(movie_title, number_of_recommendations=5):
    titles = df["title"].tolist()

    # Case-insensitive exact match
    title_lookup = {title.lower(): title for title in titles}
    query = movie_title.strip().lower()

    if query not in title_lookup:
        matches = get_close_matches(movie_title, titles, n=3, cutoff=0.4)
        if matches:
            print("Movie not found exactly. Did you mean:", ", ".join(matches))
        else:
            print("Movie not found. Try a title shown in the dataset.")
        return pd.DataFrame(columns=["Recommended Movie", "Similarity Score"])

    selected_title = title_lookup[query]
    movie_index = df.index[df["title"] == selected_title][0]

    scores = list(enumerate(similarity_matrix[movie_index]))
    scores = sorted(scores, key=lambda item: item[1], reverse=True)

    # Skip the first result, which is the selected movie itself
    recommendations = []
    for index, score in scores:
        if index == movie_index:
            continue
        recommendations.append({
            "Recommended Movie": df.iloc[index]["title"],
            "Similarity Score": round(float(score), 3),
            "Genres": df.iloc[index]["genres"]
        })
        if len(recommendations) >= number_of_recommendations:
            break

    return pd.DataFrame(recommendations)

## 7. Test the recommendation system

Change `selected_movie` to any movie title in the dataset. Try titles such as `The Dark Knight`, `Toy Story`, `Inception`, or `Avatar`.


In [47]:
selected_movie = ""
recommendations = recommend_movies(selected_movie, number_of_recommendations=5)
display(recommendations)

Movie not found exactly. Did you mean: Rent, Yentl, Teeth


,Recommended Movie,Similarity Score


## 8. Try another movie

Run this cell with a different title to see how the recommendations change.


In [ ]:
recommend_movies("Toy Story", number_of_recommendations=5)

## 9. Optional: Recommend movies from your own CSV dataset

To use a larger dataset, load a CSV containing a movie title column and one or more descriptive columns. The code below assumes the file is named `movies.csv` and includes columns such as `title`, `genres`, `overview`, `keywords`, `cast`, and `director`.

If your dataset uses different column names, update `rename_map`. After loading the CSV, rerun the preparation, TF-IDF, similarity, and recommendation cells above. The sample dataset remains the default so the notebook runs immediately.


## 10. Result and conclusion

**Result:** The content-based movie recommendation system was implemented using Python, TF-IDF vectorization, and cosine similarity. It ranks movies by how similar their descriptive features are to a selected movie.

**Conclusion:** The project demonstrates how data preprocessing and text similarity can be used to build a basic recommendation system. The quality of recommendations depends on the completeness and quality of the movie metadata. This is a content-based system; it does not learn from user ratings or viewing history.

**Limitations**
- The embedded dataset is small and intended for demonstration.
- Similarity scores indicate similarity of text features, not guaranteed user preference.
- A larger, well-cleaned dataset can improve recommendation variety and usefulness.
